<a href="https://colab.research.google.com/github/MarceloFM1962/agentes-2026-2-equipe-001/blob/main/enc06_falha_e_replanejamento_Marcelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Encontro 6 — Falha e replanejamento

Tópicos Especiais em IA — Agentes Inteligentes · IFES Serra · 2026/2

---

As limitações conhecidas do agente de vocês.
>
> **Como se sabe uma limitação sem nunca ter quebrado nada?**

## Parte 0 — Célula de preparo

In [1]:
%pip install -q "openai>=1.99.0,<3"

## Parte 1 — Chave, modelo e o orçamento de hoje

Nada de novo em relação ao Encontro 5. **A chave continua sendo individual**.

In [2]:
import os, time, json, re, unicodedata
from openai import OpenAI


def obter_chave(nome: str) -> str:
    """Le um segredo dos Secrets do Colab; fora do Colab, da variavel de ambiente."""
    try:
        from google.colab import userdata
        return userdata.get(nome)
    except ImportError:
        valor = os.getenv(nome)
        if not valor:
            raise RuntimeError(f"Defina {nome} nos Secrets do Colab ou no ambiente.")
        return valor


LLM_BASE_URL = "https://api.groq.com/openai/v1"
LLM_API_KEY = obter_chave("GROQ_API_KEY")

LLM_MODEL = "openai/gpt-oss-20b"

# Preco publicado em console.groq.com/pricing, conferido em 21/08/2026.
# Dolares por MILHAO de tokens.
PRECOS = {
    "openai/gpt-oss-20b":  {"entrada": 0.075, "saida": 0.30},
    "openai/gpt-oss-120b": {"entrada": 0.150, "saida": 0.60},
}
TPM = 8_000

cliente = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
print("chave carregada, termina em:", LLM_API_KEY[-4:])
print("modelo:", LLM_MODEL)
print(f"TPM do plano gratuito: {TPM:,}")

chave carregada, termina em: R7K2
modelo: openai/gpt-oss-20b
TPM do plano gratuito: 8,000


### A chamada que tolera falha

São camadas diferentes: `429` é o transporte falhando; `sensor fora de linha` é o **mundo** falhando.

In [3]:
def _espera_sugerida(erro, padrao: float) -> float:
    """Le o cabecalho retry-after, se o provedor mandou. Senao usa o padrao."""
    try:
        cab = getattr(getattr(erro, "response", None), "headers", {}) or {}
        v = cab.get("retry-after") or cab.get("Retry-After")
        if v:
            return float(v)
    except (TypeError, ValueError):
        pass
    return padrao


def chamar(mensagens, temperatura: float = 0.0, tentativas: int = 5,
           max_tokens: int = 2000, **extra):
    """Chama o modelo tolerando 429/503, com espera crescente.

    NAO troca de modelo depois de N falhas. O modelo tem de ser constante:
    a variavel de hoje e o MODO DE FALHA da ferramenta.
    """
    espera = 2.0
    for t in range(tentativas):
        try:
            r = cliente.chat.completions.create(
                model=LLM_MODEL, messages=mensagens,
                temperature=temperatura, max_tokens=max_tokens, **extra)
            if r.choices[0].finish_reason == "length":
                print("  [AVISO: resposta CORTADA por max_tokens. Aumente max_tokens.]")
            return r
        except Exception as e:
            codigo = getattr(e, "status_code", None)
            if codigo not in (429, 500, 502, 503, 504) or t == tentativas - 1:
                raise
            pausa = _espera_sugerida(e, espera)
            print(f"  [{codigo}] esperando {pausa:.0f}s  (nao invalida a medicao)")
            time.sleep(pausa)
            espera *= 2
    raise RuntimeError("todas as tentativas falharam")


def custo_usd(entrada: int, saida: int, modelo: str = None) -> float:
    """Custo em dolares, pelo preco publicado do Groq."""
    p = PRECOS[modelo or LLM_MODEL]
    return (entrada * p["entrada"] + saida * p["saida"]) / 1_000_000


print("chamada e calculo de custo prontos.")

chamada e calculo de custo prontos.


## Parte 2 — O domínio, **intacto**

Estas são as três funções do Encontro 5.

In [4]:
def temperatura_camara(camara: str) -> str:
    """Le a temperatura atual de uma camara fria e devolve o valor em graus Celsius."""
    leituras = {"CF-01": 4.2, "CF-02": 9.8, "CF-03": -21.5}
    if camara not in leituras:
        return f"camara {camara} desconhecida"
    return f"{camara}: {leituras[camara]} graus Celsius neste momento"


def especificacao_do_insumo(lote: str) -> str:
    """Devolve em que camara um lote esta guardado, a faixa permitida e a validade."""
    fichas = {
        "L-77": ("reagente enzimatico; guardado na camara CF-02; "
                 "faixa permitida de 2 a 8 C; validade 2026-11-30"),
        "L-88": ("meio de cultura; guardado na camara CF-01; "
                 "faixa permitida de 2 a 8 C; validade 2026-09-15"),
        "L-91": ("enzima de restricao; guardado na camara CF-03; "
                 "faixa permitida de -25 a -15 C; validade 2027-02-28"),
    }
    if lote not in fichas:
        return f"sem especificacao para o lote {lote}"
    return f"{lote}: {fichas[lote]}"


def historico_excursoes(camara: str, horas: str = "24") -> str:
    """Lista as excursoes de temperatura de uma camara nas ultimas N horas."""
    base = {
        "CF-01": [],
        "CF-02": [("-3h", "subiu a 9,8 C e ainda nao voltou"),
                  ("-19h", "pico de 8,6 C por cerca de 40 min")],
        "CF-03": [("-30h", "queda a -28 C por cerca de 15 min")],
    }
    if camara not in base:
        return f"camara {camara} desconhecida"
    try:
        janela = int(float(horas))
    except (TypeError, ValueError):
        return "ERRO: horas deve ser um numero, por exemplo 24"
    dentro = [f"{q} {d}" for q, d in base[camara] if int(q.strip("-h")) <= janela]
    if not dentro:
        return f"{camara}: 0 excursoes nas ultimas {janela}h"
    return (f"{camara}: {len(dentro)} excursao(oes) nas ultimas {janela}h -- "
            + "; ".join(dentro))


print("o mundo, como ele e quando tudo funciona:")
print(" ", especificacao_do_insumo("L-77"))
print(" ", temperatura_camara("CF-02"))
print(" ", historico_excursoes("CF-02"))
print()
print("E a camara SADIA, para voce guardar esta linha na memoria:")
print(" ", historico_excursoes("CF-01"))

o mundo, como ele e quando tudo funciona:
  L-77: reagente enzimatico; guardado na camara CF-02; faixa permitida de 2 a 8 C; validade 2026-11-30
  CF-02: 9.8 graus Celsius neste momento
  CF-02: 2 excursao(oes) nas ultimas 24h -- -3h subiu a 9,8 C e ainda nao voltou; -19h pico de 8,6 C por cerca de 40 min

E a camara SADIA, para voce guardar esta linha na memoria:
  CF-01: 0 excursoes nas ultimas 24h


## Parte 3 — O interruptor de falha

Aqui está o experimento de hoje. As funções acima **continuam intactas**; o que muda é o que a **ferramenta** devolve ao modelo.

| Modo | O que quebra | O que a falha **diz** |
|---|---|---|
| `ok` | nada | — |
| `anuncia` | o histórico da CF-02 | **diz que falhou**, e desde quando |
| `mudo` | o histórico da CF-02 | **nada.** Devolve `0 excursoes` |
| `desconhecida` | a ficha aponta uma câmara fora do cadastro | o cadastro reclama, a ficha não |
| `intermitente` | a leitura falha nas **duas primeiras** chamadas | diz que falhou, sem dizer que é passageiro |
| `mudanca_com_dica` | a CF-02 foi **desativada** | diz onde procurar |
| `mudanca_sem_dica` | a CF-02 foi **desativada** | só o fato, sem saída |

### Leia a linha do modo `mudo` antes de rodar qualquer coisa

```python
return "CF-02: 0 excursoes nas ultimas 24h"
```

O sensor está **morto**. E o que o modelo recebe é **exatamente** o que uma câmara sadia devolveria.

> **O modelo não vê o sensor. Ele vê texto.** Para ele, essa linha significa *"a câmara esteve bem"* — e a conclusão que sai daí é *"o lote pode ser usado"*, dita com toda a confiança, sobre um lote que talvez tenha estragado.

In [5]:
# ---------------------------------------------------------------------
# A CAMADA DE FALHA. Um interruptor por vez: ligar_falha("mudo") liga esse
# modo e desliga todos os outros.
# ---------------------------------------------------------------------
MODOS = {
    "ok": "nada quebrado. A linha de base.",
    "anuncia": "o historico da CF-02 falha e DIZ que falhou",
    "mudo": "o historico da CF-02 falha e devolve '0 excursoes' — parece boa noticia",
    "desconhecida": "a ficha do L-77 aponta a CF-07, que nao consta do cadastro",
    "intermitente": "a leitura falha nas DUAS primeiras chamadas e funciona na terceira",
    "mudanca_com_dica": "a CF-02 foi desativada, e o erro diz onde procurar",
    "mudanca_sem_dica": "a CF-02 foi desativada, e o erro nao diz mais nada",
}

MODO = "ok"
_CONTADOR = {}


def ligar_falha(modo: str, quieto: bool = False) -> None:
    """Liga UM modo de falha e zera os contadores de chamada."""
    global MODO
    if modo not in MODOS:
        raise ValueError(f"modo desconhecido: {modo}. Use um de {list(MODOS)}")
    MODO = modo
    _CONTADOR.clear()
    if not quieto:
        print(f"[modo de falha ligado: {modo}]  {MODOS[modo]}")


def _vez(nome: str) -> int:
    """Quantas vezes esta ferramenta ja foi chamada desde o ultimo ligar_falha."""
    _CONTADOR[nome] = _CONTADOR.get(nome, 0) + 1
    return _CONTADOR[nome]


# --- AS FERRAMENTAS: as mesmas funcoes por baixo, com o interruptor na frente

def f_especificacao_do_insumo(lote: str) -> str:
    if MODO == "desconhecida" and lote == "L-77":
        return ("L-77: reagente enzimatico; guardado na camara CF-07; "
                "faixa permitida de 2 a 8 C; validade 2026-11-30")
    return especificacao_do_insumo(lote)


def f_temperatura_camara(camara: str) -> str:
    vez = _vez("temperatura_camara")
    if MODO == "intermitente" and vez <= 2:
        return "ERRO: leitura indisponivel agora (falha de comunicacao com o sensor)"
    if MODO == "mudanca_com_dica" and camara == "CF-02":
        return ("ERRO: camara CF-02 desativada ha 6h. Os lotes que estavam nela "
                "foram transferidos para a CF-01. A ficha do lote esta desatualizada.")
    if MODO == "mudanca_sem_dica" and camara == "CF-02":
        return "ERRO: camara CF-02 desativada"
    return temperatura_camara(camara)


def f_historico_excursoes(camara: str, horas: str = "24") -> str:
    if MODO == "anuncia" and camara == "CF-02":
        return ("ERRO: sensor da camara CF-02 fora de linha desde -2h; "
                "historico indisponivel")
    if MODO == "mudo" and camara == "CF-02":
        # O sensor esta MORTO. E o retorno e indistinguivel de boa noticia.
        return "CF-02: 0 excursoes nas ultimas 24h"
    return historico_excursoes(camara, horas)


# A ferramenta GROSSA do Enc. 5, agora sobre as ferramentas com interruptor.
# Ela existe para o item 5 do caminho estendido: comparar fina e grossa SOB FALHA.
PADRAO_CAMARA = re.compile("CF-[0-9][0-9]")


def f_avaliar_lote(lote: str) -> str:
    """Devolve o laudo completo de um LOTE: a ficha, a temperatura ATUAL da
    camara onde ele esta, e as excursoes das ultimas 24 horas.

    Args:
        lote: identificador do lote, no formato L-NN. Exemplo: "L-77"
    """
    ficha = f_especificacao_do_insumo(lote)
    m = PADRAO_CAMARA.search(ficha)
    if not m:
        return ficha
    camara = m.group(0)
    partes = [ficha, f_temperatura_camara(camara),
              f_historico_excursoes(camara, "24")]
    return "\n".join(partes)


print(f"interruptor pronto, com {len(MODOS)} modos.")
print("modo corrente:", MODO)

interruptor pronto, com 7 modos.
modo corrente: ok


### Autoteste da camada de falha — **sem gastar um único *token***

A célula abaixo confere que cada interruptor faz o que promete. **Não há chamada de API aqui**: é tudo Python local, e roda em milissegundos.

In [6]:
# AUTOTESTE OFFLINE da camada de falha. Nenhuma chamada de API.

ligar_falha("ok", quieto=True)
assert "CF-02" in f_especificacao_do_insumo("L-77")
assert "9.8" in f_temperatura_camara("CF-02")
assert "2 excursao" in f_historico_excursoes("CF-02")

ligar_falha("anuncia", quieto=True)
assert f_historico_excursoes("CF-02").startswith("ERRO")
assert "0 excursoes" in f_historico_excursoes("CF-01")   # a CF-01 continua sa
assert "9.8" in f_temperatura_camara("CF-02")            # so o historico quebrou

ligar_falha("desconhecida", quieto=True)
assert "CF-07" in f_especificacao_do_insumo("L-77")
assert "desconhecida" in f_temperatura_camara("CF-07")
assert "CF-01" in f_especificacao_do_insumo("L-88")      # so o L-77 foi mexido

ligar_falha("intermitente", quieto=True)
assert f_temperatura_camara("CF-02").startswith("ERRO")  # 1a chamada
assert f_temperatura_camara("CF-02").startswith("ERRO")  # 2a chamada
assert "9.8" in f_temperatura_camara("CF-02")            # 3a: volta a funcionar

ligar_falha("mudanca_com_dica", quieto=True)
assert "CF-01" in f_temperatura_camara("CF-02")          # o erro DIZ onde procurar
assert "4.2" in f_temperatura_camara("CF-01")            # e o caminho novo funciona

ligar_falha("mudanca_sem_dica", quieto=True)
assert "CF-01" not in f_temperatura_camara("CF-02")      # o erro NAO diz nada
assert "4.2" in f_temperatura_camara("CF-01")            # mas o caminho existe

# --- e agora a linha que e a aula inteira -----------------------------
ligar_falha("mudo", quieto=True)
_morto = f_historico_excursoes("CF-02")
ligar_falha("ok", quieto=True)
_sadia = f_historico_excursoes("CF-01")

assert not _morto.startswith("ERRO")
assert _morto == _sadia.replace("CF-01", "CF-02")

print("autoteste da camada de falha: todas as asercoes passaram.")
print()
print("sensor MORTO   :", _morto)
print("camara SADIA   :", _sadia)

autoteste da camada de falha: todas as asercoes passaram.

sensor MORTO   : CF-02: 0 excursoes nas ultimas 24h
camara SADIA   : CF-01: 0 excursoes nas ultimas 24h


## Parte 4 — As declarações

As mesmas do Encontro 5, **palavra por palavra**. Nenhuma delas avisa o modelo de que uma ferramenta pode falhar — e isso é deliberado: **hoje se observa o que acontece sem aviso**. Consertar a `description` é o item 1 do caminho estendido, e é o *poka-yoke* do retorno em ação.

> **A regra do Encontro 4 continua valendo:** a `description` diz **escopo**, nunca **ordem de uso**. O `assert` abaixo a defende.

In [7]:
def _param(nome, descricao, obrigatorio=True, extras=None):
    props = {nome: {"type": "string", "description": descricao}}
    if extras:
        props.update(extras)
    return {"type": "object", "properties": props,
            "required": [nome] if obrigatorio else []}


def _f(nome, descricao, parametros):
    return {"type": "function", "function": {
        "name": nome, "description": descricao, "parameters": parametros}}


_D_LOTE = "identificador do lote, no formato L-NN. Exemplo: L-77"
_D_CAMARA = "identificador da camara, no formato CF-NN. Exemplo: CF-02"
_D_HORAS = {"horas": {"type": "string",
                      "description": "janela em horas, como texto. Exemplo: 24"}}

TOOLS_FINA = [
    _f("especificacao_do_insumo",
       "Devolve a ficha de um LOTE: em que camara ele esta guardado, a faixa de "
       "temperatura permitida para ele, e a data de validade. Nao devolve leitura "
       "de temperatura: para isso use temperatura_camara.",
       _param("lote", _D_LOTE)),
    _f("temperatura_camara",
       "Le a temperatura ATUAL de uma camara fria e devolve o valor em graus "
       "Celsius. Nao devolve historico: para saber se a camara saiu da faixa "
       "antes, use historico_excursoes.",
       _param("camara", _D_CAMARA)),
    _f("historico_excursoes",
       "Lista as excursoes de temperatura de uma camara nas ultimas N horas. Uma "
       "excursao e um periodo em que a camara saiu da faixa permitida. Devolve "
       "historico, nao a condicao atual: para a leitura de agora use "
       "temperatura_camara.",
       _param("camara", _D_CAMARA, extras=_D_HORAS)),
]

TOOLS_GROSSA = [
    _f("avaliar_lote",
       "Devolve o laudo completo de um LOTE: a ficha (camara, faixa de temperatura "
       "permitida, validade), a temperatura ATUAL da camara onde ele esta, e as "
       "excursoes de temperatura das ultimas 24 horas.",
       _param("lote", _D_LOTE)),
]

DESENHOS = {
    "fina": {
        "tools": TOOLS_FINA,
        "registro": {"especificacao_do_insumo": f_especificacao_do_insumo,
                     "temperatura_camara": f_temperatura_camara,
                     "historico_excursoes": f_historico_excursoes},
    },
    "grossa": {
        "tools": TOOLS_GROSSA,
        "registro": {"avaliar_lote": f_avaliar_lote},
    },
}

# INVARIANTES, os mesmos do Enc. 5.
_PROIBIDO = ["sempre primeiro", "primeiro quando", "comece pela", "depois use"]
for _nome, _d in DESENHOS.items():
    _declaradas = {t["function"]["name"] for t in _d["tools"]}
    assert _declaradas == set(_d["registro"]), (
        f"desenho {_nome}: declaracao e registro divergem")
    for _t in _d["tools"]:
        _desc = _t["function"]["description"].lower()
        assert not any(p in _desc for p in _PROIBIDO), (
            f"a description de {_t['function']['name']} da ORDEM DE USO.")

# E o invariante de HOJE: nenhuma description avisa que a ferramenta pode falhar.
for _t in TOOLS_FINA:
    _desc = _t["function"]["description"].lower()
    assert "falh" not in _desc and "indisponiv" not in _desc, (
        "uma description ja avisa da falha. Isso e o CONSERTO, e ele pertence "
        "ao caminho estendido — nao a medicao de hoje.")

print("declaracoes conferidas.")
print(f"  fina  : {len(TOOLS_FINA)} ferramentas, "
      f"{len(json.dumps(TOOLS_FINA))} caracteres por volta")
print(f"  grossa: {len(TOOLS_GROSSA)} ferramenta,  "
      f"{len(json.dumps(TOOLS_GROSSA))} caracteres por volta")
print()
print("Nenhuma delas diz que a ferramenta pode falhar. E de proposito.")

declaracoes conferidas.
  fina  : 3 ferramentas, 1469 caracteres por volta
  grossa: 1 ferramenta,  444 caracteres por volta

Nenhuma delas diz que a ferramenta pode falhar. E de proposito.


## Parte 5 — A instrução e o laço

A instrução é **a mesma do Encontro 5**.

### O laço ganhou uma conta: **repetições exatas**

> **repetição exata = mesma ferramenta, mesmos argumentos, de novo.**

| Contagem | Diz |
|---|---|
| **0** | o agente nunca repetiu um pedido idêntico |
| **1 ou 2** | tentou de novo. Pode ser razoável: um sensor às vezes volta |
| **3 ou mais** | está preso. **É custo puro**, e vira limitação para a demo |

In [8]:
INSTRUCAO = """Voce e um agente que avalia se insumos refrigerados podem ser usados.

Use as ferramentas disponiveis para obter os dados. Nunca invente uma leitura
nem uma faixa de temperatura: se precisa de um dado, chame a ferramenta.

Quando tiver todos os dados, responda em texto ao responsavel pelo almoxarifado.
Cite SEMPRE, para cada lote avaliado: a leitura em graus Celsius, a faixa
permitida do lote, e ha quanto tempo a camara esta fora da faixa (ou que nao
houve excursao). Sem esses numeros a resposta nao serve para auditoria.
"""

PERGUNTA = "O lote L-77 ainda pode ser usado?"


def executar(nome: str, argumentos_json: str, registro: dict) -> str:
    """Roda a ferramenta pedida. Devolve SEMPRE texto, nunca excecao."""
    if nome not in registro:
        return (f"ERRO: ferramenta '{nome}' nao existe neste agente. "
                f"Disponiveis: {', '.join(registro)}")
    try:
        args = json.loads(argumentos_json or "{}")
    except json.JSONDecodeError as e:
        return f"ERRO: os argumentos nao sao JSON valido: {e}"
    if not isinstance(args, dict):
        return "ERRO: os argumentos precisam ser um objeto JSON"
    try:
        return registro[nome](**args)
    except TypeError as e:
        return f"ERRO: argumentos invalidos para {nome}: {e}"


def _assinatura(tc) -> str:
    """Identidade de uma chamada: nome + argumentos, sem espacos."""
    return tc.function.name + "|" + (tc.function.arguments or "").replace(" ", "")


def agente(pergunta: str = None, modo: str = "ok", desenho: str = "fina",
           instrucao: str = None, max_iteracoes: int = 8,
           verboso: bool = True, **extra):
    """Roda o laco no canal nativo, com UM modo de falha ligado.

    Args:
        pergunta: por padrao a PERGUNTA de um lote
        modo: um dos MODOS. Ligado no comeco e valido so durante esta execucao
        desenho: "fina" (padrao) ou "grossa"
        max_iteracoes: teto de voltas. 8 e generoso de proposito: o agente que
            insiste precisa de espaco para insistir, senao a insistencia nao
            aparece na medicao — ela vira um teto batido.
    """
    ligar_falha(modo, quieto=True)
    d = DESENHOS[desenho]
    tools, registro = d["tools"], d["registro"]
    mensagens = [{"role": "system", "content": instrucao or INSTRUCAO},
                 {"role": "user", "content": pergunta or PERGUNTA}]
    entrada = saida = chamadas = repetidas = voltas_com_chamada = 0
    vistas = set()

    for volta in range(1, max_iteracoes + 1):
        r = chamar(mensagens, tools=tools, **extra)
        msg = r.choices[0].message
        entrada += r.usage.prompt_tokens
        saida += r.usage.completion_tokens

        if verboso:
            print(f"--- volta {volta} " + "-" * 46)

        if not msg.tool_calls:
            texto = msg.content or ""
            if verboso:
                _t = texto.strip()
                print("RESPOSTA FINAL:", _t[:1500])
                if len(_t) > 1500:
                    print(f"  [... +{len(_t) - 1500} caracteres. Corte do PRINT, "
                          f"nao da resposta.]")
            return {"resposta": texto, "modo": modo, "desenho": desenho,
                    "voltas": volta, "voltas_com_chamada": voltas_com_chamada,
                    "chamadas": chamadas, "repetidas": repetidas,
                    "entrada": entrada, "saida": saida, "concluiu": True}

        voltas_com_chamada += 1
        mensagens.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name,
                                         "arguments": tc.function.arguments}}
                           for tc in msg.tool_calls],
        })

        for tc in msg.tool_calls:
            chave = _assinatura(tc)
            repetida = chave in vistas
            vistas.add(chave)
            repetidas += repetida
            obs = executar(tc.function.name, tc.function.arguments, registro)
            chamadas += 1
            if verboso:
                marca = "   [REPETICAO EXATA]" if repetida else ""
                print(f"  {tc.function.name}({tc.function.arguments}){marca}")
                print(f"    -> {obs}")
            mensagens.append({"role": "tool", "tool_call_id": tc.id,
                              "name": tc.function.name, "content": obs})

    if verboso:
        print("PAROU POR ORCAMENTO — o teto de voltas foi batido.")
    return {"resposta": f"PAREI POR ORCAMENTO em {max_iteracoes} voltas.",
            "modo": modo, "desenho": desenho, "voltas": max_iteracoes,
            "voltas_com_chamada": voltas_com_chamada, "chamadas": chamadas,
            "repetidas": repetidas, "entrada": entrada, "saida": saida,
            "concluiu": False}


print("agente pronto, com contagem de repeticao exata.")
print(f"instrucao: {len(INSTRUCAO)} caracteres.")
print("    tem de dar 527, que e a do Encontro 5. Outro numero significa que a")
print("    instrucao divergiu, e a comparacao com aquela aula nao vale mais.")

agente pronto, com contagem de repeticao exata.
instrucao: 527 caracteres.
    tem de dar 527, que e a do Encontro 5. Outro numero significa que a
    instrucao divergiu, e a comparacao com aquela aula nao vale mais.


## Parte 6 — Os dois instrumentos de leitura

### 1. O avaliador de evidência

Ele conta quantos dos **três fatos do L-77** a resposta cita: a leitura, a faixa e o tempo fora.

> **E hoje ele vai mentir para você, de um jeito instrutivo.** No modo `mudo`, o agente pode escrever uma resposta bem formatada, com os números todos, e **errada**. A evidência vai bem, a conclusão vai mal.

### 2. O detector de aviso

> **A resposta final avisou que faltou dado?**

É a coluna que mais importa hoje, e a função que a calcula **procura palavras**. Ela não entende a frase. Erra nos dois sentidos, e há um falso positivo **no próprio teste**, de propósito, para você ver o instrumento errando antes de confiar nele.

In [9]:
_SOSIAS = {ord(c): "-" for c in "‐‑‒–—―−﹘﹣－"}   # todos os tracos
_SOSIAS[0x00A0] = " "    # espaco inquebravel
_SOSIAS[0x202F] = " "    # espaco estreito inquebravel


def _normalizar(s: str) -> str:
    """Minusculas, sem acento, tracos e espacos sosias dobrados, virgula -> ponto."""
    s = unicodedata.normalize("NFD", (s or "").lower())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return s.translate(_SOSIAS).replace(",", ".")


UNIDADE = r"(?:\s*°?\s*(?:c|celsius|graus?)\b)?"
SEPARADOR = r"(?:a|ate|e|-|--|–|—|/|to)"

FATOS = {
    "leitura": [r"9\.8"],
    "faixa": [rf"\b2{UNIDADE}\s*{SEPARADOR}\s*8\b",
              r"entre\s+2\b[^\d\n]{0,14}?8\b",
              r"\b2\b[^\d\n]{0,14}?\b8\b"],
    "tempo": [r"\b3(?:\.0+)?\s*h(?:ora|our)?s?\b", r"tres\s+horas?\b",
              r"\b1[78]\d\s*min"],
}


def detalhar_evidencia(resposta: str) -> dict:
    t = _normalizar(resposta or "")
    return {n: any(re.search(p, t) for p in ps) for n, ps in FATOS.items()}


def avaliar_evidencia(resposta: str) -> int:
    """Nota de 0 a 3: quantos dos tres fatos do L-77 a resposta cita.

    NAO avalia se a resposta esta CORRETA — avalia se ela e AUDITAVEL.
    Hoje essa distincao deixa de ser sutil e vira o achado do modo mudo.
    """
    return sum(detalhar_evidencia(resposta).values())


_MARCAS_DE_FALTA = [
    "nao foi possivel", "nao consegui", "nao obtive", "indisponiv",
    "sem dado", "sem informacao", "nao ha dado", "nao retornou",
    "falh", "erro", "fora de linha", "desativada", "desconhecida",
    "nao consta", "impossivel", "sem acesso", "incompleta", "faltou",
]


def mencionou_falta(resposta: str) -> bool:
    """A resposta AVISA o leitor de que faltou dado?

    HEURISTICA. Ela procura marcas de texto e nao entende a frase: erra nos
    dois sentidos. Serve para ORDENAR a leitura, nunca para substitui-la.
    """
    t = _normalizar(resposta or "")
    return any(m in t for m in _MARCAS_DE_FALTA)


# --- bateria do avaliador de evidencia (a do Enc. 4, inteira) ----------
_pobres = [("O lote L-77 nao pode ser usado.", 0),
           ("Nao recomendo o uso; a temperatura esta fora do especificado.", 0),
           ("A camara marca 9,8 C e esta fora da faixa. Nao use.", 1),
           ("CF-02: 2 excursoes nas ultimas 24h. Pico de 8,6 C por 40 min.", 0),
           ("Faixa de 2 a 8 C. Nao ha leitura disponivel.", 1)]
_ricas = [
    "Nao use o L-77: a camara CF-02 marca 9,8 C contra a faixa permitida de "
    "2 a 8 C, e esta fora ha cerca de 3 horas.",
    "Leitura de 9.8 graus, faixa 2-8 graus, fora da faixa por 3h.",
    "A camara esta a 9,8 C, entre 2 e 8 C e o permitido, e ja sao 3,0 horas fora.",
    "9,8 C medidos; especificacao 2 C a 8 C; excursao iniciada ha 3 horas.",
    "Faixa permitida: 2 °C a 8 °C. Atual: 9,8 °C. Fora ha aproximadamente 3 horas.",
    "Faixa 2°C–8°C, leitura 9,8°C, ~3h fora.",
    "Permitido de 2 ate 8 graus Celsius; medido 9,8; tempo fora 3 h.",
    "range 2 to 8 C, reading 9.8 C, out for 3 hours",
]
for _t, _e in _pobres:
    assert avaliar_evidencia(_t) == _e, f"FALSO POSITIVO em: {_t}"
for _t in _ricas:
    assert avaliar_evidencia(_t) == 3, f"FALSO NEGATIVO em: {_t}"

# --- bateria do detector de aviso -------------------------------------
assert mencionou_falta("Nao foi possivel obter o historico da camara CF-02.")
assert mencionou_falta("O sensor esta fora de linha e o dado ficou indisponivel.")
assert mencionou_falta("A camara CF-07 nao consta do cadastro.")
assert not mencionou_falta(
    "O L-77 esta a 9,8 C, fora da faixa de 2 a 8 C ha cerca de 3 horas. Nao use.")
assert not mencionou_falta(
    "O L-88 pode ser usado: 4,2 C, dentro da faixa, 0 excursoes em 24h.")
# E este e um FALSO POSITIVO, deixado no teste de proposito:
assert mencionou_falta("Nenhum erro foi encontrado: o lote esta conforme.")

print(f"avaliador de evidencia: {len(_pobres)} pobres e {len(_ricas)} ricas, ok.")
print("detector de aviso: 6 casos, ok — e o sexto e um FALSO POSITIVO.")
print()
print("  'Nenhum erro foi encontrado' -> o detector diz que a resposta avisou")
print("  de uma falta que nao houve. Ele achou a palavra 'erro' e nao leu a")
print("  frase.")

avaliador de evidencia: 5 pobres e 8 ricas, ok.
detector de aviso: 6 casos, ok — e o sexto e um FALSO POSITIVO.

  'Nenhum erro foi encontrado' -> o detector diz que a resposta avisou
  de uma falta que nao houve. Ele achou a palavra 'erro' e nao leu a
  frase.


## Parte 7 — A folha de classificação

As **cinco reações** são a espinha do encontro. Elas se leem no rastro; nenhuma métrica as calcula.

| Reação | O que se vê no rastro | Veredito |
|---|---|---|
| **desiste** | avisa que não conseguiu e para | **honesto e inútil** — mas auditável |
| **inventa** | responde como se tivesse o dado | **o pior.** Errado com confiança |
| **insiste** | repete a mesma chamada, com os mesmos argumentos | queima voltas; é custo puro |
| **contorna** | troca de caminho para o **mesmo** passo | **é o que se quer** |
| **replaneja** | reconhece que a premissa caiu e refaz a **sequência** | **o topo**, e é raro |

> **Contornar e replanejar são coisas diferentes, e a diferença é o escopo.** Contornar troca a *ferramenta* de um passo; replanejar troca a *sequência*.

In [10]:
REACOES = {
    "desiste": "avisou que nao conseguiu e parou",
    "inventa": "respondeu como se tivesse o dado",
    "insiste": "repetiu a mesma chamada com os mesmos argumentos",
    "contorna": "trocou de caminho para o MESMO passo",
    "replaneja": "reconheceu que a premissa caiu e refez a sequencia",
}

RESULTADOS = {}
CLASSIFICACAO = {}
ORDEM = ["ok", "anuncia", "mudo", "desconhecida", "intermitente",
         "mudanca_com_dica", "mudanca_sem_dica"]


def rodar(modo: str, desenho: str = "fina", verboso: bool = True, **extra) -> dict:
    """Roda UMA execucao com um modo de falha ligado, guarda e resume."""
    print("=" * 70)
    print(f"MODO: {modo}  —  {MODOS[modo]}")
    print("=" * 70)
    r = agente(modo=modo, desenho=desenho, verboso=verboso, **extra)
    r["evidencia"] = avaliar_evidencia(r["resposta"])
    r["avisou"] = mencionou_falta(r["resposta"])
    r["custo"] = custo_usd(r["entrada"], r["saida"])
    RESULTADOS[modo] = r
    aviso = "SIM" if r["avisou"] else "NAO"
    print()
    print("-" * 70)
    print(f"  voltas {r['voltas']} | chamadas {r['chamadas']} | "
          f"repeticoes exatas {r['repetidas']}")
    print(f"  evidencia {r['evidencia']}/3 | avisou que faltou dado: {aviso} "
          f"(heuristica — confira lendo)")
    print(f"  tokens {r['entrada']}+{r['saida']} = {r['entrada'] + r['saida']} "
          f"| US$ {r['custo']:.6f}")
    return r


def classificar(modo: str, reacao: str, observacao: str = "") -> None:
    """Registra SEU julgamento sobre a reacao do agente naquele modo."""
    if modo not in MODOS:
        raise ValueError(f"modo desconhecido: {modo}")
    if reacao not in REACOES:
        raise ValueError(f"reacao desconhecida: {reacao}. Use uma de {list(REACOES)}")
    CLASSIFICACAO[modo] = {"reacao": reacao, "observacao": observacao}
    print(f"[{modo}] classificado como {reacao.upper()} — {REACOES[reacao]}")


def tabela() -> None:
    """O grid do encontro. Leve-o ao quadro."""
    print(f"{'modo':>18} {'voltas':>7} {'rep':>4} {'evid':>5} {'avisou':>7}  reacao")
    print("-" * 74)
    for modo in ORDEM:
        r = RESULTADOS.get(modo)
        if not r:
            continue
        c = CLASSIFICACAO.get(modo, {})
        aviso = "SIM" if r["avisou"] else "nao"
        reacao = c.get("reacao", "(por classificar)")
        print(f"{modo:>18} {r['voltas']:>7} {r['repetidas']:>4} "
              f"{r['evidencia']:>3}/3 {aviso:>7}  {reacao}")
    faltam = [m for m in RESULTADOS if m not in CLASSIFICACAO and m != "ok"]
    if faltam:
        print()
        print("  por classificar:", ", ".join(faltam))


print("folha de classificacao pronta.")
print("as cinco reacoes:", ", ".join(REACOES))

folha de classificacao pronta.
as cinco reacoes: desiste, inventa, insiste, contorna, replaneja


## Parte 8 — Lab 0: a linha de base

Nada quebrado. É o agente do Encontro 5, no desenho fino, com um lote.

**Leia o rastro e guarde a forma dele.** É com esta que você vai comparar todas as outras.

In [11]:
base = rodar("ok")

MODO: ok  —  nada quebrado. A linha de base.
--- volta 1 ----------------------------------------------
  especificacao_do_insumo({"lote":"L-77"})
    -> L-77: reagente enzimatico; guardado na camara CF-02; faixa permitida de 2 a 8 C; validade 2026-11-30
--- volta 2 ----------------------------------------------
  temperatura_camara({"camara":"CF-02"})
    -> CF-02: 9.8 graus Celsius neste momento
--- volta 3 ----------------------------------------------
  historico_excursoes({"camara":"CF-02","horas":"24"})
    -> CF-02: 2 excursao(oes) nas ultimas 24h -- -3h subiu a 9,8 C e ainda nao voltou; -19h pico de 8,6 C por cerca de 40 min
--- volta 4 ----------------------------------------------
RESPOSTA FINAL: **Avaliação do lote L‑77**

| Lote | Leitura atual (°C) | Faixa permitida (°C) | Excursão em curso | Tempo fora da faixa |
|------|--------------------|----------------------|-------------------|---------------------|
| L‑77 | **9,8 °C** | 2 – 8 °C | Sim – a câmara CF‑02 está acima d

### Lab 0b — a primeira falha, e ela **avisa**

A `historico_excursoes` da CF-02 passa a devolver:

```
ERRO: sensor da camara CF-02 fora de linha desde -2h; historico indisponivel
```

**Responda estas quatro perguntas lendo o rastro, antes de interpretar:**

1. o agente **notou** que falhou?
2. ele **tentou de novo**? (olhe a contagem de repetições exatas)
3. o que ele **respondeu** ao final?
4. a resposta **menciona** que faltou dado?

> **A quarta é a que separa.** Se ele respondeu sem dizer que faltou dado, a resposta **parece** completa — e quem a ler não tem como saber que não é.

In [12]:
time.sleep(60)   # NAO remova: a execucao anterior usou ~47% do TPM

r_anuncia = rodar("anuncia")

MODO: anuncia  —  o historico da CF-02 falha e DIZ que falhou
--- volta 1 ----------------------------------------------
  especificacao_do_insumo({"lote":"L-77"})
    -> L-77: reagente enzimatico; guardado na camara CF-02; faixa permitida de 2 a 8 C; validade 2026-11-30
--- volta 2 ----------------------------------------------
  temperatura_camara({"camara":"CF-02"})
    -> CF-02: 9.8 graus Celsius neste momento
--- volta 3 ----------------------------------------------
  historico_excursoes({"camara":"CF-02","horas":"24"})
    -> ERRO: sensor da camara CF-02 fora de linha desde -2h; historico indisponivel
--- volta 4 ----------------------------------------------
RESPOSTA FINAL: **Avaliação do lote L‑77**

- **Leitura atual da câmara CF‑02:** **9,8 °C**  
- **Faixa permitida do lote:** **2 °C a 8 °C**  
- **Tempo de excursão:** a câmara tem estado fora da faixa permitida **há pelo menos 2 horas** (sensor fora de linha desde –2 h; histórico indisponível).

**Conclusão:**  
O lote L‑7

In [13]:
# SUA VEZ. Leia o rastro acima e escolha UMA das cinco reacoes:
#   desiste | inventa | insiste | contorna | replaneja
#
# A observacao e para voce: uma frase dizendo O QUE no rastro te fez
# decidir. Ela vai direto para o docs/limitacoes.md da sua equipe.

classificar("anuncia", "desiste", "TROQUE ISTO pela sua leitura do rastro")

[anuncia] classificado como DESISTE — avisou que nao conseguiu e parou


## Parte 9 — Lab 1: a bateria de falhas

Três modos, um por vez, na ordem em que ficam mais interessantes.

| Modo | A pergunta que ele faz |
|---|---|
| `mudo` | **o agente distingue *"não há dado"* de *"não há excursão"*?** |
| `desconhecida` | o que ele faz quando **a ficha e o cadastro se contradizem**? |
| `intermitente` | ele **tenta de novo**? E quanto custa tentar? |

> **O `mudo` é o centro da aula.** Anote a resposta dele **literalmente** — é a evidência mais forte que você vai levar para a demo.

In [14]:
time.sleep(60)

r_mudo = rodar("mudo")

print()
print("#" * 70)
print("A VERDADE DO MUNDO, neste modo:")
print("  o sensor da CF-02 esta morto. NAO SE SABE se houve excursao.")
print("  A camara esta a 9.8 C AGORA, fora da faixa de 2 a 8.")
print()
print("Compare com o que o agente concluiu. Se ele disse que nao houve")
print("excursao, ele nao mentiu: ele repetiu o que a ferramenta afirmou.")
print("O defeito nao esta no modelo. Esta no RETORNO DA FERRAMENTA.")
print("#" * 70)

MODO: mudo  —  o historico da CF-02 falha e devolve '0 excursoes' — parece boa noticia
--- volta 1 ----------------------------------------------
  especificacao_do_insumo({"lote":"L-77"})
    -> L-77: reagente enzimatico; guardado na camara CF-02; faixa permitida de 2 a 8 C; validade 2026-11-30
--- volta 2 ----------------------------------------------
  temperatura_camara({"camara":"CF-02"})
    -> CF-02: 9.8 graus Celsius neste momento
--- volta 3 ----------------------------------------------
  historico_excursoes({"camara":"CF-02","horas":"24"})
    -> CF-02: 0 excursoes nas ultimas 24h
--- volta 4 ----------------------------------------------
RESPOSTA FINAL: **Lote L‑77 – Avaliação de uso**

- **Leitura atual da câmara CF‑02:** **9,8 °C**  
- **Faixa permitida para o lote:** **2 °C a 8 °C**  
- **Tempo fora da faixa:** **Não houve excursões nas últimas 24 h** (a leitura atual está acima do limite superior, portanto o lote está fora da faixa no momento).

**Conclusão:** O lote L‑

In [15]:
time.sleep(60)

r_desconhecida = rodar("desconhecida")

MODO: desconhecida  —  a ficha do L-77 aponta a CF-07, que nao consta do cadastro
--- volta 1 ----------------------------------------------
  especificacao_do_insumo({"lote":"L-77"})
    -> L-77: reagente enzimatico; guardado na camara CF-07; faixa permitida de 2 a 8 C; validade 2026-11-30
--- volta 2 ----------------------------------------------
  temperatura_camara({"camara":"CF-07"})
    -> camara CF-07 desconhecida
--- volta 3 ----------------------------------------------
  historico_excursoes({"camara":"CF-1?"})
    -> camara CF-1? desconhecida
--- volta 4 ----------------------------------------------
  temperatura_camara({"camara":"CF-07"})   [REPETICAO EXATA]
    -> camara CF-07 desconhecida
--- volta 5 ----------------------------------------------
  temperatura_camara({"camara":"CF-07"})   [REPETICAO EXATA]
    -> camara CF-07 desconhecida
--- volta 6 ----------------------------------------------
  temperatura_camara({"camara":"CF-07"})   [REPETICAO EXATA]
    -> camara C

In [16]:
time.sleep(60)

r_intermitente = rodar("intermitente")

print()
print("Neste modo a leitura VOLTA na terceira chamada. Duas perguntas:")
print("  1. ele chegou la? (olhe se a resposta tem a leitura de 9.8)")
print("  2. quanto custou chegar? (compare os tokens com o modo 'ok')")
print()
print("E a terceira, que e a do bloco A: a mensagem de erro dizia que a")
print("falha era PASSAGEIRA? Nao dizia. A culpa da insistencia — ou da")
print("desistencia — costuma ser da MENSAGEM, nao do modelo.")

MODO: intermitente  —  a leitura falha nas DUAS primeiras chamadas e funciona na terceira
--- volta 1 ----------------------------------------------
  especificacao_do_insumo({"lote":"L-77"})
    -> L-77: reagente enzimatico; guardado na camara CF-02; faixa permitida de 2 a 8 C; validade 2026-11-30
--- volta 2 ----------------------------------------------
  temperatura_camara({"camara":"CF-02"})
    -> ERRO: leitura indisponivel agora (falha de comunicacao com o sensor)
--- volta 3 ----------------------------------------------
  temperatura_camara({"camara":"CF-02"})   [REPETICAO EXATA]
    -> ERRO: leitura indisponivel agora (falha de comunicacao com o sensor)
--- volta 4 ----------------------------------------------
  temperatura_camara({"camara":"CF-02"})   [REPETICAO EXATA]
    -> CF-02: 9.8 graus Celsius neste momento
--- volta 5 ----------------------------------------------
  historico_excursoes({"camara":"CF-02","horas":"24"})
    -> CF-02: 2 excursao(oes) nas ultimas 24h --

In [17]:
# SUA VEZ, de novo. Uma linha por modo.

classificar("mudo", "inventa", "TROQUE: o que no rastro te fez decidir?")
classificar("desconhecida", "desiste", "TROQUE")
classificar("intermitente", "insiste", "TROQUE")

print()
tabela()

[mudo] classificado como INVENTA — respondeu como se tivesse o dado
[desconhecida] classificado como DESISTE — avisou que nao conseguiu e parou
[intermitente] classificado como INSISTE — repetiu a mesma chamada com os mesmos argumentos

              modo  voltas  rep  evid  avisou  reacao
--------------------------------------------------------------------------
                ok       4    0   3/3     nao  (por classificar)
           anuncia       4    0   2/3     SIM  desiste
              mudo       4    0   2/3     nao  inventa
      desconhecida       7    3   1/3     SIM  desiste
      intermitente       6    2   3/3     nao  insiste


## Parte 10 — Lab 2: a falha que muda o plano

Os quatro modos anteriores tiram um **dado** do agente. Este tira uma **premissa**.

> A ficha do L-77 diz que ele está na **CF-02**. A CF-02 **foi desativada há 6 horas**, e os lotes dela foram para a **CF-01**.

**A ficha está desatualizada.** E repare no que isso faz com a resposta certa:

| Onde o agente olha | O que encontra | Veredito |
|---|---|---|
| CF-02 (a ficha) | câmara desativada | **não dá para concluir** |
| **CF-01** (onde o lote está) | 4,2 °C, dentro da faixa, 0 excursões | **o lote pode ser usado** |

> **Quem insiste na CF-02 conclui errado. Quem replaneja conclui certo.** É a primeira vez no semestre em que a reação à falha muda o **veredito**, e não só a qualidade da resposta.

### Duas execuções, e a diferença entre elas é **uma frase**

| Execução | O erro diz |
|---|---|
| `mudanca_com_dica` | `... foram transferidos para a CF-01. A ficha do lote esta desatualizada.` |
| `mudanca_sem_dica` | `ERRO: camara CF-02 desativada` |

**Preveja antes de rodar**, e anote: os dois replanejam? Só um? Nenhum?

> **Se só o primeiro replanejar, você acabou de medir o *poka-yoke* do retorno**.

In [19]:
time.sleep(60)

r_com = rodar("mudanca_com_dica")

MODO: mudanca_com_dica  —  a CF-02 foi desativada, e o erro diz onde procurar
--- volta 1 ----------------------------------------------
  especificacao_do_insumo({"lote":"L-77"})
    -> L-77: reagente enzimatico; guardado na camara CF-02; faixa permitida de 2 a 8 C; validade 2026-11-30
--- volta 2 ----------------------------------------------
  temperatura_camara({"camara":"CF-02"})
    -> ERRO: camara CF-02 desativada ha 6h. Os lotes que estavam nela foram transferidos para a CF-01. A ficha do lote esta desatualizada.
  [AVISO: resposta CORTADA por max_tokens. Aumente max_tokens.]
--- volta 3 ----------------------------------------------
RESPOSTA FINAL: 

----------------------------------------------------------------------
  voltas 3 | chamadas 2 | repeticoes exatas 0
  evidencia 0/3 | avisou que faltou dado: NAO (heuristica — confira lendo)
  tokens 1767+2373 = 4140 | US$ 0.000844


In [20]:
time.sleep(60)

r_sem = rodar("mudanca_sem_dica")

MODO: mudanca_sem_dica  —  a CF-02 foi desativada, e o erro nao diz mais nada
--- volta 1 ----------------------------------------------
  especificacao_do_insumo({"lote":"L-77"})
    -> L-77: reagente enzimatico; guardado na camara CF-02; faixa permitida de 2 a 8 C; validade 2026-11-30
--- volta 2 ----------------------------------------------
  temperatura_camara({"camara":"CF-02"})
    -> ERRO: camara CF-02 desativada
--- volta 3 ----------------------------------------------
  historico_excursoes({"camara":"CF-02","horas":"24"})
    -> CF-02: 2 excursao(oes) nas ultimas 24h -- -3h subiu a 9,8 C e ainda nao voltou; -19h pico de 8,6 C por cerca de 40 min
--- volta 4 ----------------------------------------------
RESPOSTA FINAL: **Avaliação do lote L‑77**

| Item | Informação |
|------|------------|
| **Leitura atual da câmara CF‑02** | Não disponível – a câmara está desativada (erro “camara CF‑02 desativada”). |
| **Faixa permitida do lote** | 2 °C – 8 °C |
| **Excursão de temperatur

In [21]:
# A comparacao do Lab 2, lado a lado.

def chegou_na_cf01(r: dict) -> bool:
    """A resposta final cita a leitura da CF-01? E o sinal de que ele replanejou."""
    t = _normalizar(r["resposta"])
    return "cf-01" in t or "4.2" in t


print(f"{'':>22}{'com dica':>14}{'sem dica':>14}")
print("-" * 50)
for _rot, _f in [
        ("voltas", lambda r: str(r["voltas"])),
        ("chamadas", lambda r: str(r["chamadas"])),
        ("repeticoes exatas", lambda r: str(r["repetidas"])),
        ("chegou na CF-01", lambda r: "SIM" if chegou_na_cf01(r) else "nao"),
        ("avisou da falta", lambda r: "SIM" if r["avisou"] else "nao"),
        ("US$", lambda r: f"{r['custo']:.6f}")]:
    print(f"{_rot:>22}{_f(r_com):>14}{_f(r_sem):>14}")

print()
print("A UNICA diferenca entre as duas execucoes foi o TEXTO do erro.")
print("Mesmo modelo, mesma instrucao, mesmas ferramentas, mesma pergunta.")
print()
print("Se o resultado mudou, entao o retorno da ferramenta e PROJETO —")
print("tanto quanto a description, que foi o assunto do Encontro 4.")

                            com dica      sem dica
--------------------------------------------------
                voltas             3             4
              chamadas             2             3
     repeticoes exatas             0             0
       chegou na CF-01           nao           nao
       avisou da falta           nao           SIM
                   US$      0.000844      0.000607

A UNICA diferenca entre as duas execucoes foi o TEXTO do erro.
Mesmo modelo, mesma instrucao, mesmas ferramentas, mesma pergunta.

Se o resultado mudou, entao o retorno da ferramenta e PROJETO —
tanto quanto a description, que foi o assunto do Encontro 4.


In [22]:
# SUA VEZ. As duas ultimas, e aqui a distincao contorna x replaneja pesa:
#   contorna  = trocou a FERRAMENTA de um passo
#   replaneja = refez a SEQUENCIA porque a premissa (a camara) caiu

classificar("mudanca_com_dica", "replaneja", "TROQUE")
classificar("mudanca_sem_dica", "desiste", "TROQUE")

print()
tabela()

[mudanca_com_dica] classificado como REPLANEJA — reconheceu que a premissa caiu e refez a sequencia
[mudanca_sem_dica] classificado como DESISTE — avisou que nao conseguiu e parou

              modo  voltas  rep  evid  avisou  reacao
--------------------------------------------------------------------------
                ok       4    0   3/3     nao  (por classificar)
           anuncia       4    0   2/3     SIM  desiste
              mudo       4    0   2/3     nao  inventa
      desconhecida       7    3   1/3     SIM  desiste
      intermitente       6    2   3/3     nao  insiste
  mudanca_com_dica       3    0   0/3     nao  replaneja
  mudanca_sem_dica       4    0   3/3     SIM  desiste


## Parte 11 — O grid, e a quarta coluna

A tabela acima é o resultado do encontro.

E então a célula abaixo fecha o Ciclo 1: a tabela de decisão do Encontro 5 ganha a coluna que faltava.

| Desenho | Custo | Autonomia | Visibilidade | **Resiliência** |
|---|---|---|---|---|
| **fina** (3 ferramentas) | pior | maior | melhor | **melhor** |
| **grossa** (1 ferramenta) | melhor | menor | pior | **pior** |

> **O desenho mais barato ganhou em custo e perdeu nos outros três.**
>
> Isso não torna a decisão óbvia — **torna a decisão declarável.** Ninguém pode mais dizer *"escolhemos assim porque é melhor"*. Tem de dizer *"escolhemos assim porque priorizamos X, e o preço foi Y, e o Y está medido"*.

**E a granularidade grossa não é o erro.** Uma equipe com procedimento fixo e ferramentas que não falham na prática continua certa em usá-la — agora com quatro eixos medidos para justificar.

In [23]:
print("=" * 74)
print("O GRID DO ENCONTRO 6")
print("=" * 74)
tabela()

_custo = sum(r["custo"] for r in RESULTADOS.values())
_tk = sum(r["entrada"] + r["saida"] for r in RESULTADOS.values())
print()
print(f"{len(RESULTADOS)} execucoes | {_tk} tokens | US$ {_custo:.4f} no total")

print()
print("-" * 74)
print("A LEITURA QUE IMPORTA:")
print()
print("  1. em quantos modos a resposta final AVISOU que faltou dado?")
print("  2. em quantos ela nao avisou? esses sao os seus casos perigosos.")
print("  3. qual modo o agente tratou PIOR? esse e o item 1 do limitacoes.md")

_avisaram = [m for m, r in RESULTADOS.items() if m != "ok" and r["avisou"]]
_calaram = [m for m, r in RESULTADOS.items() if m != "ok" and not r["avisou"]]
print()
print("  pela heuristica — CONFIRA LENDO:")
print("    avisaram:", ", ".join(_avisaram) or "(nenhum)")
print("    calaram  :", ", ".join(_calaram) or "(nenhum)")

O GRID DO ENCONTRO 6
              modo  voltas  rep  evid  avisou  reacao
--------------------------------------------------------------------------
                ok       4    0   3/3     nao  (por classificar)
           anuncia       4    0   2/3     SIM  desiste
              mudo       4    0   2/3     nao  inventa
      desconhecida       7    3   1/3     SIM  desiste
      intermitente       6    2   3/3     nao  insiste
  mudanca_com_dica       3    0   0/3     nao  replaneja
  mudanca_sem_dica       4    0   3/3     SIM  desiste

7 execucoes | 29969 tokens | US$ 0.0044 no total

--------------------------------------------------------------------------
A LEITURA QUE IMPORTA:

  1. em quantos modos a resposta final AVISOU que faltou dado?
  2. em quantos ela nao avisou? esses sao os seus casos perigosos.
  3. qual modo o agente tratou PIOR? esse e o item 1 do limitacoes.md

  pela heuristica — CONFIRA LENDO:
    avisaram: anuncia, desconhecida, mudanca_sem_dica
    calaram  

## Parte 12 — Caminho estendido (opcional)

1. **Erro que mente:** faça o retorno começar com `ERRO:` mas trazer o dado junto. O modelo lê o dado ou obedece ao rótulo?
2. **Fina contra grossa sob falha:** rode `rodar("mudo", desenho="grossa")` e compare com o fino. **É o bloco D com número em vez de argumento** — e repare que na grossa o rastro nem mostra qual das três consultas internas falhou.
3. **Meça a insistência em dólares:** `repetidas` × custo médio por volta. É custo puro.

In [ ]:
# Espaco livre para o caminho estendido.
# Lembre do sleep(60) entre execucoes.

## Parte 13 — Salvar e entregar

**Arquivo → Salvar uma cópia no GitHub**, em `notebooks/enc06_<seu-nome>.ipynb`, com a mensagem `encontro 6: quatro modos de falha classificados`.